In [ ]:
import pandas as pd


Loading and understanding Data


In [ ]:
raw_df = pd.read_csv("/content/spam_or_not_spam.csv")
df = raw_df.copy()

In [ ]:
df.head(10)

,email,label
0,date wed NUMBER aug NUMBER NUMBER NUMBER NUMB...,0
1,martin a posted tassos papadopoulos the greek ...,0
2,man threatens explosion in moscow thursday aug...,0
3,klez the virus that won t die already the most...,0
4,in adding cream to spaghetti carbonara which ...,0
5,i just had to jump in here as carbonara is on...,0
6,the scotsman NUMBER august NUMBER playboy want...,0
7,martin adamson wrote isn t it just basically a...,0
8,the scotsman thu NUMBER aug NUMBER meaningful ...,0
9,i have been trying to research via sa mirrors ...,0


In [ ]:
df.tail(10)

,email,label
2990,get NUMBER free vhs or dvds click hyperlink h...,1
2991,get NUMBER free vhs or dvds click hyperlink h...,1
2992,wealth without risk discover the best kept se...,1
2993,attn sir madan strictly confidential i am plea...,1
2994,from mr desmond stevens urgent assistance you...,1
2995,abc s good morning america ranks it the NUMBE...,1
2996,hyperlink hyperlink hyperlink let mortgage le...,1
2997,thank you for shopping with us gifts for all ...,1
2998,the famous ebay marketing e course learn to s...,1
2999,hello this is chinese traditional 子 件 NUMBER世...,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   email   2999 non-null   object
 1   label   3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB


Data Cleaning

In [ ]:
#Handle missing values
print(df.isnull().sum())
#no. of records where data is missing

email    1
label    0
dtype: int64


In [ ]:
df=df.dropna()

In [ ]:
#removing duplicate records
print(df.duplicated().sum())

127


In [ ]:
df=df.drop_duplicates()

In [ ]:
#convert text to lowercase
df['email']=df['email'].str.lower()

In [ ]:
#Tokenisation
import nltk
nltk.download('punkt_tab')
df['email'] = df['email'].apply(nltk.word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#remove stopwords
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df['email'] = df['email'].apply(lambda x: ' '.join([word for word in x if word not in stop_words]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df

,email,label
0,date wed number aug number number number numbe...,0
1,martin posted tassos papadopoulos greek sculpt...,0
2,man threatens explosion moscow thursday august...,0
3,klez virus die already prolific virus ever kle...,0
4,adding cream spaghetti carbonara effect pasta ...,0
...,...,...
2995,abc good morning america ranks number christma...,1
2996,hyperlink hyperlink hyperlink let mortgage len...,1
2997,thank shopping us gifts occasions free gift nu...,1
2998,famous ebay marketing e course learn sell comp...,1


In [ ]:
#Lemmatization
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df['email'] = df['email'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))



[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
from numpy import vectorize
#vectorisation
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['email'])

In [ ]:
y  = df['label']

Splitting


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

Single Neuron Neural network


In [ ]:
# Convert sparse matrices to dense NumPy arrays if they are sparse
from scipy.sparse import issparse

if issparse(X_train):
    X_train = X_train.toarray()
if issparse(X_test):
    X_test = X_test.toarray()

In [ ]:
import numpy as np
np.random.seed(42)

n_features = X_train.shape[1]

W = np.random.randn(n_features) * 0.01
b = 0

lr = 0.01

In [ ]:
def sigmoid(z):

    return 1 / (1 + np.exp(-z))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# '0' - 2500 , '1' - 500
spam_weight = 5   # start here
epochs = 1000
losses = []
test_accuracies = []

for i in range(epochs):

    #Forward Pass
    z = np.dot(X_train, W) + b
    y_pred = sigmoid(z)

    #loss derivative (cross entropy)
    dL_dz = (y_pred - y_train) * (spam_weight*y_train + (1 - y_train))


    #Gradients
    m = X_train.shape[0]
    dL_dW = np.dot(X_train.T, dL_dz) / m
    dL_db = np.mean(dL_dz)


    #Gradient descents
    W -= lr * dL_dW
    b -= lr * dL_db

    loss = -np.mean(spam_weight*y_train*np.log(y_pred+1e-9) +
    (1-y_train)*np.log(1-y_pred+1e-9))

    losses.append(loss)

    # ✅ ADD ACCURACY CODE RIGHT HERE
    z_test = np.dot(X_test, W) + b
    y_pred_test = sigmoid(z_test)

    # Convert probabilities to binary predictions (0 or 1)
    y_pred_binary = (y_pred_test >= 0.5).astype(int)


    acc = accuracy_score(y_test, y_pred_binary)

    test_accuracies.append(acc)


    if i % 10 == 0:

        print("Loss:", loss)


plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(losses)
plt.title("Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")

plt.subplot(1,2,2)
plt.plot(test_accuracies)
plt.title("Test Accuracy Curve")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")

plt.show()



Loss: 1.1053776572705696
Loss: 1.1043797929134505
Loss: 1.103404804403361
Loss: 1.1024509340303694
Loss: 1.101516565335786
Loss: 1.10060021180993
Loss: 1.0997005064818528
Loss: 1.0988161923329063
Loss: 1.097946113470856
Loss: 1.0970892070057567
Loss: 1.0962444955730781
Loss: 1.0954110804535677
Loss: 1.0945881352430864
Loss: 1.0937749000291723
Loss: 1.0929706760343374
Loss: 1.0921748206891677
Loss: 1.0913867431011235
Loss: 1.090605899887565
Loss: 1.0898317913439826
Loss: 1.0890639579206656
Loss: 1.0883019769831401
Loss: 1.0875454598336527
Loss: 1.0867940489727657
Loss: 1.0860474155817814
Loss: 1.0853052572082589
Loss: 1.084567295638275
Loss: 1.0838332749404094
Loss: 1.0831029596676116
Loss: 1.0823761332042323
Loss: 1.081652596246509
Loss: 1.0809321654057402
Loss: 1.080214671924254
Loss: 1.0794999604950533
Loss: 1.0787878881767792
Loss: 1.0780783233962867
Loss: 1.0773711450317587
Loss: 1.0766662415698525
Loss: 1.0759635103308955
Loss: 1.0752628567566367
Loss: 1.0745641937554926
Loss: 1.0

In [ ]:
print(np.sum((y_pred_test > 0.4) & (y_pred_test < 0.6)))


560


In [ ]:
print(np.min(y_pred_test), np.max(y_pred_test))


0.39209321862367763 0.6068912025886497


In [ ]:
print(y_train.mean())


0.14671310404875926


Testing

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred_binary)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred_binary)
print("Confusion Matrix:\n", cm)

cr = classification_report(y_test, y_pred_binary)
print("Classification Report:\n", cr)

Accuracy: 0.8434782608695652
Confusion Matrix:
 [[485   0]
 [ 90   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      1.00      0.92       485
           1       0.00      0.00      0.00        90

    accuracy                           0.84       575
   macro avg       0.42      0.50      0.46       575
weighted avg       0.71      0.84      0.77       575



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Train accuracy


In [ ]:
train_preds = (sigmoid(np.dot(X_train, W) + b) > 0.5).astype(int)
print(accuracy_score(y_train, train_preds))


0.8558989986939486
